## Example 1: JSON-RPC stream writer

In [ ]:
from pylsp_jsonrpc.streams import JsonRpcStreamWriter
import json
import io

# Example 1: JSON-RPC stream writer
print("Testing python-lsp-jsonrpc stream writer:")

# Create output stream
output = io.BytesIO()
writer = JsonRpcStreamWriter(output)
print(f"✓ Created writer: {type(writer).__name__}")

# Write JSON-RPC request
request = {"jsonrpc": "2.0", "method": "initialize", "id": 1, "params": {}}
writer.write(request)
print(f"✓ Wrote request: {request['method']}")

# Write JSON-RPC notification
notification = {"jsonrpc": "2.0", "method": "textDocument/didOpen", "params": {}}
writer.write(notification)
print(f"✓ Wrote notification: {notification['method']}")

# Check output
output.seek(0)
content = output.read().decode('utf-8')
print(f"✓ Total bytes: {len(content)}")
print(f"✓ Content-Length headers: {content.count('Content-Length:')}")
# Each message is a Content-Length header, a blank line, then that many bytes of JSON
header, _, body = output.getvalue().partition(b"\r\n\r\n")
length = int(header.split(b"\r\n")[0].split(b": ")[1])
assert content.count('Content-Length:') == 2
assert json.loads(body[:length]) == request

output.close()

## Example 2: Writing and reading back a message stream

In [ ]:
from pylsp_jsonrpc.streams import JsonRpcStreamWriter, JsonRpcStreamReader
import json
import io

# Example 2: Writing and reading back a message stream
print("Testing python-lsp-jsonrpc streams:")

# Create output stream
output = io.BytesIO()
writer = JsonRpcStreamWriter(output)
print(f"✓ Created writer: {type(writer)}")

# Write JSON-RPC messages
messages = [
    {"jsonrpc": "2.0", "method": "initialize", "id": 1, "params": {}},
    {"jsonrpc": "2.0", "method": "textDocument/didOpen", "params": {"uri": "file:///test.py"}},
    {"jsonrpc": "2.0", "result": {"capabilities": {}}, "id": 1}
]

for msg in messages:
    writer.write(msg)
    print(f"✓ Wrote message: {msg.get('method') or 'response'}")

# Read written content
output.seek(0)
content = output.read().decode('utf-8')
print(f"✓ Total bytes written: {len(content)}")
print(f"✓ Messages: {content.count('Content-Length:')}")
# JsonRpcStreamReader parses the same framing back into messages
reader = JsonRpcStreamReader(io.BytesIO(output.getvalue()))
received = []
reader.listen(received.append)   # returns at the end of the stream
print(f"✓ Read back {len(received)} messages")
assert received == messages


output.close()

## Example 3: Message types and an Endpoint request handler

In [ ]:
from pylsp_jsonrpc.streams import JsonRpcStreamWriter
import json
import io

# Example 3: Message types and an Endpoint request handler
print("Testing python-lsp-jsonrpc message types:")

output = io.BytesIO()
writer = JsonRpcStreamWriter(output)

# Request with ID
request = {"jsonrpc": "2.0", "method": "textDocument/hover", "id": 1, "params": {"uri": "file:///test.py"}}
writer.write(request)
print(f"✓ Request: {request['method']}, ID={request['id']}")

# Notification (no ID)
notification = {"jsonrpc": "2.0", "method": "textDocument/didChange", "params": {}}
writer.write(notification)
print(f"✓ Notification: {notification['method']}, no ID")

# Response
response = {"jsonrpc": "2.0", "result": {"contents": "hover text"}, "id": 1}
writer.write(response)
print(f"✓ Response: ID={response['id']}, has result")

# Check stream
output.seek(0)
content = output.read().decode('utf-8')
print(f"✓ Stream bytes: {len(content)}")
assert content.count('Content-Length:') == 3

# An Endpoint dispatches a request to a handler method and sends back the response
from pylsp_jsonrpc.dispatchers import MethodDispatcher
from pylsp_jsonrpc.endpoint import Endpoint

class HoverServer(MethodDispatcher):
    # The method textDocument/hover is dispatched to m_text_document__hover
    def m_text_document__hover(self, uri=None):
        assert uri == "file:///test.py"
        return {"contents": "hover text"}

sent = []
endpoint = Endpoint(HoverServer(), sent.append)
endpoint.consume(request)
endpoint.shutdown()
print(f"✓ Endpoint response: {sent}")
assert sent == [response]

output.close()

## Example 4: JSON-RPC protocol messages

In [ ]:
from pylsp_jsonrpc.streams import JsonRpcStreamReader, JsonRpcStreamWriter
import json
import io

# Example 4: JSON-RPC protocol messages
print("Testing python-lsp-jsonrpc protocol messages:")

output = io.BytesIO()
writer = JsonRpcStreamWriter(output)

# Different message types
message_types = [
    ("Request", {"jsonrpc": "2.0", "method": "test", "id": 1, "params": {}}),
    ("Notification", {"jsonrpc": "2.0", "method": "notify", "params": {}}),
    ("Response", {"jsonrpc": "2.0", "result": {"data": "ok"}, "id": 1}),
    ("Error", {"jsonrpc": "2.0", "error": {"code": -32601, "message": "Not found"}, "id": 1})
]

for name, msg in message_types:
    writer.write(msg)
    has_id = 'id' in msg
    has_method = 'method' in msg
    has_result = 'result' in msg
    has_error = 'error' in msg
    
    print(f"✓ {name}:")
    print(f"  Has ID: {has_id}, Method: {has_method}, Result: {has_result}, Error: {has_error}")
# (has id, has method, has result, has error) for request, notification, response and error
flags = [('id' in msg, 'method' in msg, 'result' in msg, 'error' in msg) for _, msg in message_types]
assert flags == [(True, True, False, False), (False, True, False, False),
                 (True, False, True, False), (True, False, False, True)]
received = []
JsonRpcStreamReader(io.BytesIO(output.getvalue())).listen(received.append)
assert received == [msg for _, msg in message_types]


output.close()

## Example 5: Error responses

In [ ]:
from pylsp_jsonrpc.streams import JsonRpcStreamReader, JsonRpcStreamWriter
import json
import io

# Example 5: Error responses
print("Testing python-lsp-jsonrpc error handling:")

output = io.BytesIO()
writer = JsonRpcStreamWriter(output)

# Success response
success = {"jsonrpc": "2.0", "result": {"status": "ok"}, "id": 1}
writer.write(success)
print(f"✓ Success response: ID={success['id']}")

# Error response - method not found
error1 = {"jsonrpc": "2.0", "error": {"code": -32601, "message": "Method not found"}, "id": 2}
writer.write(error1)
print(f"✓ Error -32601: {error1['error']['message']}")

# Error response - invalid params
error2 = {"jsonrpc": "2.0", "error": {"code": -32602, "message": "Invalid params"}, "id": 3}
writer.write(error2)
print(f"✓ Error -32602: {error2['error']['message']}")

# Check output
output.seek(0)
content = output.read().decode('utf-8')
print(f"✓ Total messages: {content.count('Content-Length:')}")
received = []
JsonRpcStreamReader(io.BytesIO(output.getvalue())).listen(received.append)
assert [msg.get("error", {}).get("code") for msg in received] == [None, -32601, -32602]

# The library's exception classes carry the standard JSON-RPC error codes
import logging
from pylsp_jsonrpc.dispatchers import MethodDispatcher
from pylsp_jsonrpc.endpoint import Endpoint
from pylsp_jsonrpc.exceptions import JsonRpcInvalidParams, JsonRpcMethodNotFound

assert (JsonRpcMethodNotFound.CODE, JsonRpcInvalidParams.CODE) == (-32601, -32602)

# An Endpoint answers a request for a method it has no handler for with a -32601 error.
# The endpoint also logs the failed lookup; raise its log level so the log stays quiet here.
endpoint_log = logging.getLogger("pylsp_jsonrpc.endpoint")
previous_level = endpoint_log.level
endpoint_log.setLevel(logging.CRITICAL)
try:
    sent = []
    endpoint = Endpoint(MethodDispatcher(), sent.append)
    endpoint.consume({"jsonrpc": "2.0", "method": "unknown/method", "id": 4, "params": {}})
    endpoint.shutdown()
finally:
    endpoint_log.setLevel(previous_level)
print(f"✓ Unknown method: {sent}")
assert len(sent) == 1 and sent[0]["id"] == 4 and sent[0]["error"]["code"] == -32601

output.close()